# 03 · Camada Gold — Carregamento (L)

A camada gold contém as tabelas dimensionais e de factos do data mart final.

**Estratégias SCD utilizadas:**
- **SCD Tipo 1** (overwrite): `dim_date`, `dim_geography`, `dim_delivery_method` + colunas não-rastreadas das dimensões abaixo
- **SCD Tipo 2** (histórico): campos específicos em `dim_product`, `dim_customer`, `dim_employee`
  - `dim_product`: `brand`, `size`, `tax_rate`, `lead_time_days`
  - `dim_customer`: `customer_category_name`, `buying_group_name`, `is_on_credit_hold`
  - `dim_employee`: `is_salesperson`

**Ordem de carregamento** (respeitar dependências de FK):

```
silver.stg_*
  │
  ├─▶ gold.dim_date            (gerada programaticamente)
  ├─▶ gold.dim_geography       (SCD1 — TRUNCATE + reload)
  ├─▶ gold.dim_delivery_method (SCD1 — TRUNCATE + reload)
  ├─▶ gold.dim_product         (SCD2 — UPSERT híbrido)
  ├─▶ gold.dim_customer        (SCD2 — UPSERT híbrido)
  ├─▶ gold.dim_employee        (SCD2 — UPSERT híbrido)
  ├─▶ gold.fact_sales          (depende de todas as dims)
  └─▶ gold.fact_orders         (depende de todas as dims)
```

---
> **SCD Tipo 2 — estratégia:**
> 1. Fechar registos onde campos rastreados mudaram (`end_date = hoje`, `is_active = FALSE`)
> 2. Actualizar campos SCD1 in-place nos registos que ficaram correntes
> 3. Inserir novas versões + novos business keys (`start_date = hoje`, `is_active = TRUE`)

## 1. Imports e ligações

In [7]:
import pandas as pd
import numpy as np
from datetime import datetime, date, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

engine_dwh = create_engine(
    f"postgresql+psycopg2://{DWH_USER}:{DWH_PASSWORD}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"✓ Engine criado.")
print(f"  DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")
print(f"  Início: {run_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")

✓ Engine criado.
  DWH: postgres@localhost:5432/wwi_dw
  Início: 2026-04-10 16:22:09 UTC


## 2. Funções de carregamento SCD Tipo 1

In [8]:
from datetime import date as _date

def load_scd1_truncate(df: pd.DataFrame, gold_table: str, schema: str = "gold") -> int:
    """
    Recarrega uma dimensão por TRUNCATE + INSERT.
    Adequado para dimensões pequenas (ex: dim_delivery_method).
    A chave surrogate (SERIAL) é gerida pelo PostgreSQL.
    """
    with engine_dwh.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {schema}.{gold_table} RESTART IDENTITY CASCADE"))
        df.to_sql(gold_table, conn, schema=schema, if_exists="append", index=False)
    return len(df)


def upsert_scd1_bulk(df: pd.DataFrame, gold_table: str, business_key: str,
                     update_cols: list, schema: str = "gold") -> dict:
    """
    UPSERT em bulk via tabela temporária — muito mais eficiente para volumes grandes.
    1. Carrega df numa temp table
    2. INSERT ... ON CONFLICT DO UPDATE a partir da temp
    3. Devolve contagens aproximadas
    """
    tmp_table = f"_tmp_{gold_table}"
    col_list  = ", ".join(df.columns.tolist())
    set_clause = ", ".join([f"{c} = EXCLUDED.{c}" for c in update_cols])

    with engine_dwh.begin() as conn:
        # 1. Carregar na temp
        df.to_sql(tmp_table, conn, schema="public", if_exists="replace", index=False)

        # 2. Contar existentes (aproximação)
        existing = conn.execute(
            text(f"""
                SELECT COUNT(*) FROM public.{tmp_table} t
                WHERE EXISTS (
                    SELECT 1 FROM {schema}.{gold_table} g
                    WHERE g.{business_key} = t.{business_key}
                )
            """)
        ).scalar()

        # 3. UPSERT
        conn.execute(text(f"""
            INSERT INTO {schema}.{gold_table} ({col_list})
            SELECT {col_list} FROM public.{tmp_table}
            ON CONFLICT ({business_key})
            DO UPDATE SET {set_clause}
        """))

        # 4. Limpar temp
        conn.execute(text(f"DROP TABLE IF EXISTS public.{tmp_table}"))

    return {"total": len(df), "updated_approx": existing, "inserted_approx": len(df) - existing}


def upsert_scd2_bulk(df: pd.DataFrame, gold_table: str, business_key: str,
                     scd2_cols: list, scd1_cols: list, schema: str = "gold") -> dict:
    """
    SCD Tipo 2 + Tipo 1 híbrido (bulk via tabela temporária).
      - scd2_cols: alteração → fecha registo actual, insere nova versão
      - scd1_cols: alteração → actualiza in-place no registo corrente
    Colunas de controlo: start_date, end_date, is_active
    """
    tmp   = f"_tmp_{gold_table}"
    today = _date.today()
    far   = _date(9999, 12, 31)

    with engine_dwh.begin() as conn:
        # 1. Fonte → temp
        df.to_sql(tmp, conn, schema="public", if_exists="replace", index=False)

        # 2. Fechar registos correntes onde algum campo SCD2 mudou
        scd2_changed = " OR ".join([f"g.{c} IS DISTINCT FROM t.{c}" for c in scd2_cols])
        n_closed = conn.execute(text(f"""
            UPDATE {schema}.{gold_table} g
            SET end_date = :today, is_active = FALSE
            FROM public.{tmp} t
            WHERE g.{business_key} = t.{business_key}
              AND g.is_active = TRUE
              AND ({scd2_changed})
        """), {"today": today}).rowcount

        # 3. SCD1: actualizar colunas não-rastreadas nos registos que ficaram correntes
        if scd1_cols:
            scd1_set = ", ".join([f"{c} = t.{c}" for c in scd1_cols])
            n_scd1 = conn.execute(text(f"""
                UPDATE {schema}.{gold_table} g
                SET {scd1_set}
                FROM public.{tmp} t
                WHERE g.{business_key} = t.{business_key}
                  AND g.is_active = TRUE
            """)).rowcount
        else:
            n_scd1 = 0

        # 4. Inserir novas versões (fechadas + novos business keys)
        col_list = ", ".join(df.columns.tolist())
        n_inserted = conn.execute(text(f"""
            INSERT INTO {schema}.{gold_table} ({col_list}, start_date, end_date, is_active)
            SELECT {col_list}, :today, :far, TRUE
            FROM public.{tmp} t
            WHERE NOT EXISTS (
                SELECT 1 FROM {schema}.{gold_table} g
                WHERE g.{business_key} = t.{business_key}
                  AND g.is_active = TRUE
            )
        """), {"today": today, "far": far}).rowcount

        # 5. Limpar temp
        conn.execute(text(f"DROP TABLE IF EXISTS public.{tmp}"))

    return {
        "total_source":  len(df),
        "new_versions":  n_closed,
        "inserted_new":  n_inserted - n_closed,
        "scd1_updated":  n_scd1,
    }


def get_surrogate_map(gold_table: str, business_key: str, surrogate_key: str,
                      current_only: bool = False) -> dict:
    """Devolve {business_key_value: surrogate_key_value} para todos os registos da dimensão."""
    where = " WHERE is_active = TRUE" if current_only else ""
    sql = f"SELECT {business_key}, {surrogate_key} FROM gold.{gold_table}{where}"
    with engine_dwh.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    return dict(zip(df[business_key], df[surrogate_key]))


print("✓ Funções de carregamento SCD Tipo 1 e Tipo 2 definidas.")

✓ Funções de carregamento SCD Tipo 1 e Tipo 2 definidas.


## 3. `dim_date`

Gerada programaticamente a partir da amplitude de datas em `stg_invoices` e `stg_orders`.
Formato da chave: `YYYYMMDD` (INT).

In [9]:
# Determinar amplitude de datas (incluir expected_delivery_date)
sql_min_max = """
    SELECT
        LEAST(
            MIN(invoice_date),
            (SELECT MIN(order_date) FROM silver.stg_orders),
            (SELECT MIN(expected_delivery_date) FROM silver.stg_orders)
        ) AS date_min,
        GREATEST(
            MAX(invoice_date),
            (SELECT MAX(order_date) FROM silver.stg_orders),
            (SELECT MAX(expected_delivery_date) FROM silver.stg_orders)
        ) AS date_max
    FROM silver.stg_invoices
"""
with engine_dwh.connect() as conn:
    row = conn.execute(text(sql_min_max)).fetchone()

date_min = row[0] if row[0] else date(2013, 1, 1)
date_max = row[1] if row[1] else date(2016, 12, 31)

print(f"Amplitude de datas: {date_min}  →  {date_max}")

# Gerar calendário
day_names_pt = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
month_names_pt = ["", "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
                  "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]

date_range = pd.date_range(start=date_min, end=date_max, freq="D")

df_date = pd.DataFrame({"full_date": date_range.date})
df_date["date_key"]     = pd.to_datetime(df_date["full_date"]).dt.strftime("%Y%m%d").astype(int)
df_date["day"]          = pd.to_datetime(df_date["full_date"]).dt.day
df_date["month"]        = pd.to_datetime(df_date["full_date"]).dt.month
df_date["month_name"]   = df_date["month"].map(lambda m: month_names_pt[m])
df_date["quarter"]      = pd.to_datetime(df_date["full_date"]).dt.quarter
df_date["year"]         = pd.to_datetime(df_date["full_date"]).dt.year
df_date["week_of_year"] = pd.to_datetime(df_date["full_date"]).dt.isocalendar().week.astype(int)
df_date["day_of_week"]  = pd.to_datetime(df_date["full_date"]).dt.weekday + 1  # 1=Segunda
df_date["day_name"]     = (pd.to_datetime(df_date["full_date"]).dt.weekday
                           .map(lambda d: day_names_pt[d]))
df_date["is_weekend"]   = df_date["day_of_week"] >= 6
df_date["is_holiday"]   = False  # sem dados de feriados na fonte

df_date = df_date[[
    "date_key", "full_date", "day", "month", "month_name",
    "quarter", "year", "week_of_year", "day_of_week", "day_name",
    "is_weekend", "is_holiday"
]]

# Inserir na gold (TRUNCATE CASCADE + reload)
with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_date CASCADE"))
    df_date.to_sql("dim_date", conn, schema="gold", if_exists="append", index=False)

print(f"✓ gold.dim_date   — {len(df_date)} dias ({date_min} → {date_max})")
df_date.head(3)

Amplitude de datas: 2017-01-01  →  2020-06-01
✓ gold.dim_date   — 1248 dias (2017-01-01 → 2020-06-01)


,date_key,full_date,day,month,month_name,quarter,year,week_of_year,day_of_week,day_name,is_weekend,is_holiday
0,20170101,2017-01-01,1,1,Janeiro,1,2017,52,7,Domingo,True,False
1,20170102,2017-01-02,2,1,Janeiro,1,2017,1,1,Segunda,False,False
2,20170103,2017-01-03,3,1,Janeiro,1,2017,1,2,Terça,False,False


## 4. `dim_geography`

SCD Tipo 1: TRUNCATE + RELOAD (sem city_id no modelo final).

In [10]:
with engine_dwh.connect() as conn:
    df_geo = pd.read_sql(text("SELECT * FROM silver.stg_geography"), conn)

# Gold dim_geography: sem city_id e sem sales_territory
gold_geo = df_geo[["city_name", "state_province_name", "country_name"]].copy()

n = load_scd1_truncate(gold_geo, "dim_geography")
print(f"✓ gold.dim_geography — {n} registos")

# Mapa city_id → geography_key para lookups posteriores
# Juntar silver (com city_id) com gold (com geography_key) via nomes
with engine_dwh.connect() as conn:
    df_gold_geo = pd.read_sql(text("SELECT geography_key, city_name, state_province_name, country_name FROM gold.dim_geography"), conn)

geo_lookup = df_geo[["city_id", "city_name", "state_province_name", "country_name"]].merge(
    df_gold_geo, on=["city_name", "state_province_name", "country_name"], how="left"
).drop_duplicates("city_id")

geo_map = dict(zip(geo_lookup["city_id"], geo_lookup["geography_key"]))
print(f"  Mapa geography: {len(geo_map)} entradas")

✓ gold.dim_geography — 37940 registos
  Mapa geography: 37940 entradas


## 5. `dim_delivery_method`

Dimensão pequena — TRUNCATE + reload.

In [11]:
with engine_dwh.connect() as conn:
    df_dm = pd.read_sql(text("SELECT * FROM silver.stg_delivery_methods"), conn)

gold_dm = df_dm[["delivery_method_id", "delivery_method_name"]].copy()

n = load_scd1_truncate(gold_dm, "dim_delivery_method")
print(f"✓ gold.dim_delivery_method — {n} registos")

dm_map = get_surrogate_map("dim_delivery_method", "delivery_method_id", "delivery_method_key")
print(f"  Mapa delivery_method: {len(dm_map)} entradas")

✓ gold.dim_delivery_method — 10 registos
  Mapa delivery_method: 10 entradas


## 6. `dim_product`

SCD Tipo 2 nos campos: `brand`, `size`, `tax_rate`, `lead_time_days`.  
SCD Tipo 1 (overwrite) nos restantes: `stock_item_name`, `color_name`, `unit_package_name`, `outer_package_name`, `quantity_per_outer`, `is_chiller_stock`.

In [12]:
with engine_dwh.connect() as conn:
    df_prod = pd.read_sql(text("SELECT * FROM silver.stg_products"), conn)

gold_prod = df_prod[[
    "stock_item_id", "stock_item_name", "color_name",
    "unit_package_name", "outer_package_name", "brand", "size",
    "lead_time_days", "quantity_per_outer", "is_chiller_stock",
    "tax_rate"
]].copy()

scd2_cols_prod = ["brand", "size", "tax_rate", "lead_time_days"]
scd1_cols_prod = [
    "stock_item_name", "color_name",
    "unit_package_name", "outer_package_name",
    "quantity_per_outer", "is_chiller_stock"
]

result = upsert_scd2_bulk(gold_prod, "dim_product", "stock_item_id",
                          scd2_cols_prod, scd1_cols_prod)

print(f"✓ gold.dim_product — fonte={result['total_source']}  "
      f"novas_versões={result['new_versions']}  "
      f"inseridos_novos={result['inserted_new']}  "
      f"scd1_actualizados={result['scd1_updated']}")

prod_map = get_surrogate_map("dim_product", "stock_item_id", "product_key", current_only=True)
print(f"  Mapa product (correntes): {len(prod_map)} entradas")

✓ gold.dim_product — fonte=227  novas_versões=0  inseridos_novos=227  scd1_actualizados=0
  Mapa product (correntes): 227 entradas


## 7. `dim_customer`

SCD Tipo 2 nos campos: `customer_category_name`, `buying_group_name`, `is_on_credit_hold`.  
SCD Tipo 1 nos restantes: `customer_name`, `bill_to_customer_name`, `phone_number`, `geography_key`.  
Resolve `geography_key` via `city_id` → mapa do `dim_geography`.

In [13]:
with engine_dwh.connect() as conn:
    df_cust = pd.read_sql(text("SELECT * FROM silver.stg_customers"), conn)

# Resolver geography_key
df_cust["geography_key"] = df_cust["city_id"].map(geo_map)

no_geo = df_cust["geography_key"].isna().sum()
if no_geo:
    print(f"⚠ {no_geo} clientes sem city_id mapeado → geography_key = NULL")

df_cust["geography_key"] = df_cust["geography_key"].astype("Int64")

gold_cust = df_cust[[
    "customer_id", "customer_name", "customer_category_name",
    "buying_group_name", "bill_to_customer_name",
    "phone_number", "geography_key", "is_on_credit_hold"
]].copy()

scd2_cols_cust = ["customer_category_name", "buying_group_name", "is_on_credit_hold"]
scd1_cols_cust = ["customer_name", "bill_to_customer_name", "phone_number", "geography_key"]

result = upsert_scd2_bulk(gold_cust, "dim_customer", "customer_id",
                          scd2_cols_cust, scd1_cols_cust)

print(f"✓ gold.dim_customer — fonte={result['total_source']}  "
      f"novas_versões={result['new_versions']}  "
      f"inseridos_novos={result['inserted_new']}  "
      f"scd1_actualizados={result['scd1_updated']}")

cust_map = get_surrogate_map("dim_customer", "customer_id", "customer_key", current_only=True)
print(f"  Mapa customer (correntes): {len(cust_map)} entradas")

✓ gold.dim_customer — fonte=663  novas_versões=0  inseridos_novos=663  scd1_actualizados=0
  Mapa customer (correntes): 663 entradas


## 8. `dim_employee`

SCD Tipo 2 no campo: `is_salesperson`.  
SCD Tipo 1 nos restantes: `full_name`, `preferred_name`, `geography_key`.  
Resolve `geography_key` via `city_id` → mapa do `dim_geography`.

In [14]:
with engine_dwh.connect() as conn:
    df_emp = pd.read_sql(text("SELECT * FROM silver.stg_employees"), conn)

df_emp["geography_key"] = df_emp["city_id"].map(geo_map)
df_emp["geography_key"] = df_emp["geography_key"].astype("Int64")

no_geo_emp = df_emp["geography_key"].isna().sum()
if no_geo_emp:
    print(f"⚠ {no_geo_emp} employees sem city_id mapeado → geography_key = NULL")

gold_emp = df_emp[[
    "person_id", "full_name", "preferred_name", "is_salesperson", "geography_key"
]].copy()

scd2_cols_emp = ["is_salesperson"]
scd1_cols_emp = ["full_name", "preferred_name", "geography_key"]

result = upsert_scd2_bulk(gold_emp, "dim_employee", "person_id",
                          scd2_cols_emp, scd1_cols_emp)

print(f"✓ gold.dim_employee — fonte={result['total_source']}  "
      f"novas_versões={result['new_versions']}  "
      f"inseridos_novos={result['inserted_new']}  "
      f"scd1_actualizados={result['scd1_updated']}")

emp_map = get_surrogate_map("dim_employee", "person_id", "employee_key", current_only=True)
print(f"  Mapa employee (correntes): {len(emp_map)} entradas")

⚠ 1111 employees sem city_id mapeado → geography_key = NULL
✓ gold.dim_employee — fonte=1111  novas_versões=0  inseridos_novos=1111  scd1_actualizados=0
  Mapa employee (correntes): 1111 entradas


## 9. `fact_sales`

Granularidade: **uma linha por InvoiceLine**.

Resolução de chaves surrogate:
- `date_key`                    ← `stg_invoices.invoice_date` → formato YYYYMMDD
- `customer_key`                ← `stg_invoices.customer_id` → `dim_customer`
- `product_key`                 ← `stg_invoice_lines.stock_item_id` → `dim_product`
- `salesperson_employee_key`    ← `stg_invoices.salesperson_person_id` → `dim_employee`
- `delivery_method_key`         ← `stg_invoices.delivery_method_id` → `dim_delivery_method`

Medidas:
- `quantity`, `unit_price`, `tax_amount`, `line_total_excl_tax`

In [15]:
with engine_dwh.connect() as conn:
    df_inv = pd.read_sql(text("SELECT * FROM silver.stg_invoices"), conn)
    df_il  = pd.read_sql(text("SELECT * FROM silver.stg_invoice_lines"), conn)

# Juntar invoice_lines com invoices (para obter todas as FK)
df_fs = df_il.merge(
    df_inv[["invoice_id", "customer_id", "salesperson_person_id",
            "delivery_method_id", "invoice_date", "order_id"]],
    on="invoice_id", how="left"
)

# Converter date_key
df_fs["date_key"] = pd.to_datetime(df_fs["invoice_date"]).dt.strftime("%Y%m%d").astype(int)

# Resolver chaves surrogate via mapas (vectorizado)
df_fs["customer_key"]              = df_fs["customer_id"].map(cust_map)
df_fs["product_key"]               = df_fs["stock_item_id"].map(prod_map)
df_fs["salesperson_employee_key"]  = df_fs["salesperson_person_id"].map(emp_map)
df_fs["delivery_method_key"]       = df_fs["delivery_method_id"].map(dm_map)

# Validar chaves não resolvidas
for col in ["customer_key", "product_key", "salesperson_employee_key", "delivery_method_key"]:
    nulls = df_fs[col].isna().sum()
    if nulls:
        print(f"⚠ fact_sales.{col}: {nulls} chaves não resolvidas")

# Montar tabela de factos
fact_sales = df_fs[[
    "customer_key", "date_key", "delivery_method_key", "product_key",
    "salesperson_employee_key",
    "invoice_id", "order_id",
    "quantity", "unit_price", "tax_amount", "line_total_excl_tax"
]].copy()

# Converter surrogate keys para Int64 (suportam NULL)
for col in ["customer_key", "product_key", "salesperson_employee_key", "delivery_method_key"]:
    fact_sales[col] = fact_sales[col].astype("Int64")

# Carregar para gold (TRUNCATE + reload — facto é sempre recarregado)
with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_sales"))
    fact_sales.to_sql("fact_sales", conn, schema="gold", if_exists="append", index=False)

print(f"✓ gold.fact_sales — {len(fact_sales)} linhas")
fact_sales.head(3)

✓ gold.fact_sales — 228265 linhas


,customer_key,date_key,delivery_method_key,product_key,salesperson_employee_key,invoice_id,order_id,quantity,unit_price,tax_amount,line_total_excl_tax
0,438,20170101,3,68,2,1,1,10,230.0,345.00,2300.0
1,135,20170101,3,50,8,2,2,9,13.0,17.55,117.0
2,135,20170101,3,10,8,2,2,9,32.0,43.20,288.0


## 10. `fact_orders`

Granularidade: **uma linha por OrderLine**.

Resolução de chaves surrogate:
- `order_date_key`              ← `stg_orders.order_date` → formato YYYYMMDD
- `expected_delivery_date_key`  ← `stg_orders.expected_delivery_date` → formato YYYYMMDD
- `customer_key`                ← `stg_orders.customer_id` → `dim_customer`
- `product_key`                 ← `stg_order_lines.stock_item_id` → `dim_product`
- `salesperson_employee_key`    ← `stg_orders.salesperson_person_id` → `dim_employee`

Medidas:
- `ordered_quantity`, `picked_quantity`, `backordered_quantity`

In [16]:
with engine_dwh.connect() as conn:
    df_ord = pd.read_sql(text("SELECT * FROM silver.stg_orders"), conn)
    df_ol  = pd.read_sql(text("SELECT * FROM silver.stg_order_lines"), conn)

# Juntar order_lines com orders
df_fo = df_ol.merge(
    df_ord[["order_id", "customer_id", "salesperson_person_id",
            "order_date", "expected_delivery_date"]],
    on="order_id", how="left"
)

# Converter date keys
df_fo["order_date_key"] = pd.to_datetime(df_fo["order_date"]).dt.strftime("%Y%m%d").astype(int)
df_fo["expected_delivery_date_key"] = pd.to_datetime(df_fo["expected_delivery_date"]).dt.strftime("%Y%m%d").astype("Int64")

# Resolver chaves surrogate
df_fo["customer_key"]              = df_fo["customer_id"].map(cust_map)
df_fo["product_key"]               = df_fo["stock_item_id"].map(prod_map)
df_fo["salesperson_employee_key"]  = df_fo["salesperson_person_id"].map(emp_map)

# Validar
for col in ["customer_key", "product_key", "salesperson_employee_key"]:
    nulls = df_fo[col].isna().sum()
    if nulls:
        print(f"⚠ fact_orders.{col}: {nulls} chaves não resolvidas")

fact_orders = df_fo[[
    "customer_key", "order_date_key", "product_key",
    "expected_delivery_date_key", "salesperson_employee_key",
    "order_id",
    "ordered_quantity", "picked_quantity", "backordered_quantity"
]].copy()

for col in ["customer_key", "product_key", "salesperson_employee_key", "expected_delivery_date_key"]:
    fact_orders[col] = fact_orders[col].astype("Int64")

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_orders"))
    fact_orders.to_sql("fact_orders", conn, schema="gold", if_exists="append", index=False)

print(f"✓ gold.fact_orders — {len(fact_orders)} linhas")
fact_orders.head(3)

✓ gold.fact_orders — 231412 linhas


,customer_key,order_date_key,product_key,expected_delivery_date_key,salesperson_employee_key,order_id,ordered_quantity,picked_quantity,backordered_quantity
0,438,20170101,68,20170102,2,1,10,10,0
1,135,20170101,50,20170102,8,2,9,9,0
2,135,20170101,10,20170102,8,2,9,9,0


## 11. Validação final do data mart

In [17]:
gold_tables = [
    "dim_date", "dim_geography", "dim_delivery_method",
    "dim_product", "dim_customer", "dim_employee",
    "fact_sales", "fact_orders",
]

rows = []
with engine_dwh.connect() as conn:
    for t in gold_tables:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM gold.{t}")).scalar()
        rows.append({"tabela": f"gold.{t}", "registos": cnt})

df_summary = pd.DataFrame(rows)
print("Contagens no gold:")
print(df_summary.to_string(index=False))

# Verificar integridade referencial básica — fact_sales
print("\nIntegridade referencial (fact_sales):")
ref_checks_sales = [
    ("customer_key",              "dim_customer",        "customer_key"),
    ("product_key",               "dim_product",         "product_key"),
    ("salesperson_employee_key",  "dim_employee",        "employee_key"),
    ("delivery_method_key",       "dim_delivery_method", "delivery_method_key"),
]
with engine_dwh.connect() as conn:
    for fk, dim, pk in ref_checks_sales:
        orphans = conn.execute(text(f"""
            SELECT COUNT(*) FROM gold.fact_sales fs
            LEFT JOIN gold.{dim} d ON fs.{fk} = d.{pk}
            WHERE fs.{fk} IS NOT NULL AND d.{pk} IS NULL
        """)).scalar()
        status = "✓" if orphans == 0 else f"⚠ {orphans} orphans"
        print(f"  {status}  fact_sales.{fk} → gold.{dim}")

# Verificar integridade referencial — fact_orders
print("\nIntegridade referencial (fact_orders):")
ref_checks_orders = [
    ("customer_key",                "dim_customer",  "customer_key"),
    ("product_key",                 "dim_product",   "product_key"),
    ("salesperson_employee_key",    "dim_employee",  "employee_key"),
    ("order_date_key",              "dim_date",      "date_key"),
    ("expected_delivery_date_key",  "dim_date",      "date_key"),
]
with engine_dwh.connect() as conn:
    for fk, dim, pk in ref_checks_orders:
        orphans = conn.execute(text(f"""
            SELECT COUNT(*) FROM gold.fact_orders fo
            LEFT JOIN gold.{dim} d ON fo.{fk} = d.{pk}
            WHERE fo.{fk} IS NOT NULL AND d.{pk} IS NULL
        """)).scalar()
        status = "✓" if orphans == 0 else f"⚠ {orphans} orphans"
        print(f"  {status}  fact_orders.{fk} → gold.{dim}")

print("\n✓ Gold concluído. Pipeline ELT completo!")
print(f"  Duração total: {(datetime.now(timezone.utc).replace(tzinfo=None) - run_at).seconds}s")

Contagens no gold:
                  tabela  registos
           gold.dim_date      1248
      gold.dim_geography     37940
gold.dim_delivery_method        10
        gold.dim_product       227
       gold.dim_customer       663
       gold.dim_employee      1111
         gold.fact_sales    228265
        gold.fact_orders    231412

Integridade referencial (fact_sales):
  ✓  fact_sales.customer_key → gold.dim_customer
  ✓  fact_sales.product_key → gold.dim_product
  ✓  fact_sales.salesperson_employee_key → gold.dim_employee
  ✓  fact_sales.delivery_method_key → gold.dim_delivery_method

Integridade referencial (fact_orders):
  ✓  fact_orders.customer_key → gold.dim_customer
  ✓  fact_orders.product_key → gold.dim_product
  ✓  fact_orders.salesperson_employee_key → gold.dim_employee
  ✓  fact_orders.order_date_key → gold.dim_date
  ✓  fact_orders.expected_delivery_date_key → gold.dim_date

✓ Gold concluído. Pipeline ELT completo!
  Duração total: 101s


## 12. Queries de exemplo — smoke tests analíticos

Verifica que o data mart responde a perguntas de negócio típicas.

In [18]:
# Vendas por ano e trimestre
sql = """
    SELECT d.year, d.quarter,
           COUNT(DISTINCT fs.invoice_id)         AS num_invoices,
           SUM(fs.quantity)                      AS total_qty,
           ROUND(SUM(fs.line_total_excl_tax), 2) AS total_revenue
    FROM   gold.fact_sales fs
    JOIN   gold.dim_date   d  ON fs.date_key = d.date_key
    GROUP  BY d.year, d.quarter
    ORDER  BY d.year, d.quarter;
"""
with engine_dwh.connect() as conn:
    df_q1 = pd.read_sql(text(sql), conn)

print("Vendas por ano/trimestre:")
df_q1

Vendas por ano/trimestre:


,year,quarter,num_invoices,total_qty,total_revenue
0,2017,1,4383,542877,10417702.35
1,2017,2,4987,657188,12546608.60
2,2017,3,4849,615365,11656799.30
3,2017,4,4548,586227,11086077.75
4,2018,1,4733,594891,11399675.95
5,2018,2,5281,671583,12952517.85
6,2018,3,5077,643445,12754759.50
7,2018,4,5212,657482,12822533.90
8,2019,1,5231,657873,13125150.15
9,2019,2,5755,701050,14069835.75


In [19]:
# Top 10 produtos por receita
sql = """
    SELECT p.stock_item_name,
           SUM(fs.quantity)                      AS total_qty,
           ROUND(SUM(fs.line_total_excl_tax), 2) AS total_revenue
    FROM   gold.fact_sales fs
    JOIN   gold.dim_product p ON fs.product_key = p.product_key
    GROUP  BY p.stock_item_name
    ORDER  BY total_revenue DESC
    LIMIT  10;
"""
with engine_dwh.connect() as conn:
    df_q2 = pd.read_sql(text(sql), conn)

print("Top 10 produtos por receita:")
df_q2

Top 10 produtos por receita:


,stock_item_name,total_qty,total_revenue
0,Air cushion machine (Blue),5849,11107251.0
1,32 mm Anti static bubble wrap (Blue) 50m,60800,6384000.0
2,10 mm Anti static bubble wrap (Blue) 50m,63930,6329070.0
3,20 mm Double sided bubble wrap 50m,57540,6214320.0
4,32 mm Double sided bubble wrap 50m,55270,6190240.0
5,10 mm Double sided bubble wrap 50m,56600,5943000.0
6,20 mm Anti static bubble wrap (Blue) 50m,56820,5795640.0
7,32 mm Anti static bubble wrap (Blue) 20m,60420,2900160.0
8,Void fill 400 L bag (White) 400L,57430,2871500.0
9,20 mm Anti static bubble wrap (Blue) 20m,54860,2468700.0


In [20]:
# Ordens pendentes (backordered) por vendedor
sql = """
    SELECT e.full_name,
           COUNT(DISTINCT fo.order_id)    AS num_orders,
           SUM(fo.ordered_quantity)       AS ordered_qty,
           SUM(fo.backordered_quantity)   AS backordered_qty,
           ROUND(
               100.0 * SUM(fo.backordered_quantity) /
               NULLIF(SUM(fo.ordered_quantity), 0), 1
           ) AS backorder_pct
    FROM   gold.fact_orders fo
    JOIN   gold.dim_employee e ON fo.salesperson_employee_key = e.employee_key
    WHERE  fo.backordered_quantity > 0
    GROUP  BY e.full_name
    ORDER  BY backordered_qty DESC
    LIMIT  10;
"""
with engine_dwh.connect() as conn:
    df_q3 = pd.read_sql(text(sql), conn)

print("Top 10 vendedores com mais backorders:")
df_q3

Top 10 vendedores com mais backorders:


,full_name,num_orders,ordered_qty,backordered_qty,backorder_pct
0,Archer Lamble,348,40677,40677,100.0
1,Jack Potter,326,39570,39570,100.0
2,Kayla Woodcock,314,37380,37380,100.0
3,Hudson Hollinworth,321,36790,36790,100.0
4,Amy Trefl,317,36548,36548,100.0
5,Hudson Onslow,293,35408,35408,100.0
6,Anthony Grosse,303,34843,34843,100.0
7,Taj Shand,297,34407,34407,100.0
8,Lily Code,290,33257,33257,100.0
9,Sophia Hinton,276,31396,31396,100.0
